# 19 - Validate Gold

Verifies Delta format, business keys, ordered-funnel invariants,
revenue reconciliation, current-customer joins, required data
quality categories, Gold lineage, and Delta `MERGE` evidence.


In [ ]:
import re

from pyspark.sql import functions as F


dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
catalog = dbutils.widgets.get("catalog").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError(f"Invalid catalog identifier: {catalog!r}")

gold_tables = [
    f"{catalog}.gold.gold_daily_funnel",
    f"{catalog}.gold.gold_revenue_daily",
    f"{catalog}.gold.gold_customer_360",
    f"{catalog}.gold.gold_data_quality_dashboard",
]

details = []
for table in gold_tables:
    exists = spark.catalog.tableExists(table)
    detail = spark.sql(f"DESCRIBE DETAIL {table}").first() if exists else None
    details.append(
        {
            "table_name": table,
            "exists": exists,
            "format": detail["format"] if detail else None,
            "num_files": detail["numFiles"] if detail else None,
        }
    )
details_df = spark.createDataFrame(details)
details_df.display()
if details_df.filter((~F.col("exists")) | (F.col("format") != "delta")).count():
    raise RuntimeError("Missing or non-Delta Gold table")

funnel = spark.table(gold_tables[0])
revenue = spark.table(gold_tables[1])
customer_360 = spark.table(gold_tables[2])
dq = spark.table(gold_tables[3])

checks = {
    "funnel_duplicate_key": funnel.groupBy("event_date").count().filter(F.col("count") > 1).count(),
    "funnel_not_monotonic": funnel.filter(
        (F.col("view_users") < F.col("add_to_cart_users"))
        | (F.col("add_to_cart_users") < F.col("purchase_users"))
    ).count(),
    "revenue_duplicate_key": revenue.groupBy("revenue_date", "country").count().filter(F.col("count") > 1).count(),
    "revenue_formula_mismatch": revenue.filter(
        F.abs(F.col("net_revenue_usd") - (F.col("gross_paid_usd") - F.col("reversed_usd"))) > F.lit(0.01)
    ).count(),
    "customer_360_duplicate_key": customer_360.groupBy("user_id").count().filter(F.col("count") > 1).count(),
    "dq_duplicate_key": dq.groupBy("quality_date", "source_name", "quality_status", "error_type").count().filter(F.col("count") > 1).count(),
    "dq_non_positive_count": dq.filter(F.col("record_count") <= 0).count(),
}

current_customers = spark.table(f"{catalog}.silver.silver_customers").filter("is_current")
checks["customer_360_missing_current"] = current_customers.select("user_id").join(
    customer_360.select("user_id"), "user_id", "left_anti"
).count()
checks["customer_360_not_current"] = customer_360.select("user_id").join(
    current_customers.select("user_id"), "user_id", "left_anti"
).count()

history = (
    spark.table(f"{catalog}.silver.silver_order_status_history")
    .filter(F.col("status").isin("paid", "cancelled", "returned"))
    .withColumn("revenue_date", F.to_date("status_ts"))
    .withColumn("country", F.coalesce("country", F.lit("UNKNOWN")))
)
expected_revenue = (
    history.groupBy("revenue_date", "country")
    .agg(
        F.sum(
            F.when(F.col("status") == "paid", F.col("subtotal_usd"))
            .when(F.col("status").isin("cancelled", "returned"), -F.col("subtotal_usd"))
            .otherwise(F.lit(0))
        ).cast("decimal(28,2)").alias("expected_net")
    )
)
checks["revenue_reconciliation_mismatch"] = (
    expected_revenue.alias("expected")
    .join(revenue.alias("actual"), ["revenue_date", "country"], "full")
    .filter(
        F.coalesce(F.abs(F.col("expected.expected_net") - F.col("actual.net_revenue_usd")), F.lit(1)) > F.lit(0.01)
    )
    .count()
)

required_error_types = {
    "parse_error",
    "missing_key",
    "invalid_timestamp",
    "schema_mismatch",
    "duplicate_key",
    "fx_rate_missing",
}
actual_error_types = {
    row.error_type
    for row in dq.filter(F.col("quality_status") == "error")
    .select("error_type").distinct().collect()
}
missing_error_types = sorted(required_error_types - actual_error_types)
checks["dq_missing_required_error_types"] = len(missing_error_types)

lineage = spark.table(f"{catalog}.control.record_lineage")
for table in gold_tables:
    checks[f"lineage_missing_{table.rsplit('.', 1)[-1]}"] = (
        0 if lineage.filter(F.col("target_table") == table).limit(1).count() else 1
    )
    merge_count = (
        spark.sql(f"DESCRIBE HISTORY {table}")
        .filter(F.col("operation") == "MERGE")
        .count()
    )
    checks[f"merge_missing_{table.rsplit('.', 1)[-1]}"] = 0 if merge_count else 1

checks_df = spark.createDataFrame(
    [{"check_name": name, "violation_count": count} for name, count in checks.items()]
)
checks_df.display()
if missing_error_types:
    print(f"Missing required DQ categories: {missing_error_types}")
if checks_df.filter(F.col("violation_count") != 0).count():
    raise RuntimeError("Gold quality validation failed")

print("Gold validation passed")
